# 3D TDGL: one notebook for prescribed voltage waveforms

**Edit the parameter cell, then Run All.** The same scalar GL–Model A solver is used for every waveform. P, self-consistent E_z and φ are shown in x–y and through the full dielectric/BTO/dielectric stack in x–z. Optional pinning sites (local α₁ scaled, e.g. × 10) are set in `defects`; strain, nucleation sites, and waveform optimization are not included.

The default studies spontaneous domain formation: zero voltage for 100 t0 from a small near-zero perturbation, small random noise (`noise_over_Ps=0.01`), and symmetric Robin polarization BCs with extrapolation length 5 nm at both BTO faces. Runs are saved under `runs/`; matching physical/numerical settings reuse their saved dynamics. Changing plot settings regenerates the figures without another simulation.


In [ ]:
from pathlib import Path
from dataclasses import asdict
import json, hashlib, os, sys, tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir())/"tdgl3d_matplotlib"))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

candidates=[Path.cwd(),Path.cwd()/"3D_TDGL",Path.cwd().parent/"3D_TDGL"]
ROOT=next((p for p in candidates if (p/"tdgl3d"/"solver.py").is_file()),None)
if ROOT is None:raise FileNotFoundError("Open this notebook from 3D_TDGL or its parent folder.")
sys.path.insert(0,str(ROOT))
from tdgl3d import Material,Grid,StackModel,NucleationSite
plt.rcParams.update({"figure.dpi":110,"axes.grid":True,"grid.alpha":.2})


## Parameters to tune

| `waveform["kind"]` | Voltage history | Main duration settings |
|---|---|---|
| `zero` | V = 0 throughout | `relaxation_t0` |
| `step_hold` | Instantaneous application, then constant V | `amplitude_V`, `hold_t0` |
| `hold_release` | Constant V, then instantaneous removal | `amplitude_V`, `hold_t0`, `relaxation_t0` |
| `triangle`, `sine` | Bipolar cycles starting at 0 | `period_t0`, `cycles`, optional `relaxation_t0` at 0 V |
| `trapezoid` | Linear rise, hold, linear fall, zero-V rest | `rise_t0`, `hold_t0`, `fall_t0`, `relaxation_t0` |
| `two_step` | Two positive levels, then fall and rest | same settings and `two_step_fraction` |
| `pund` | Continuous P, U, N, D pulse blocks | `rise_t0`, `hold_t0`, `fall_t0`, `gap_t0` |
| `custom` | Your voltage knots, linear or held between knots | custom arrays; `relaxation_t0` holds the final voltage |

For undriven domain formation choose **`zero` + `initial["kind"]="near_zero"`**. A saturated initial state can remain polarized at zero voltage. For fast waveform comparisons set `period_t0=2` and `relaxation_t0=0`. Initialization is independent of waveform choice; it is never silently reset between pulses.

Time is specified in t0 = 1/(γ|α1|); changing mobility changes its conversion to seconds. `noise_over_Ps` adds a small random noise to P: after every time step each cell receives an independent Gaussian kick of rms `noise_over_Ps`·Ps·√(Δt/t0). Uncorrelated kicks therefore add up to an rms of `noise_over_Ps`·Ps per t0. It is a seed for nucleation, not a thermal bath; it does not change the GL coefficients. `0` turns it off. The interface thickness `h_d` is for **each** dielectric; `h_d=0` gives direct ideal electrodes.

`lambda_bottom` and `lambda_top` set the polarization extrapolation lengths in metres. Each face obeys ∂P/∂n = −P/λ with outward normal n; `None` gives zero-gradient Neumann BC and `0` gives P = 0 at that face. Positive finite lengths suppress interface polarization. Both the boundary energy and its contribution to the mean rate are included.


In [ ]:
# 1. Material and capacitor geometry (SI units).
material_settings = dict(alpha_1=-1.8558e7, alpha_11=-2.097e8, alpha_111=1.0352e10,
    gamma=1e-3, G11=2e-10, G44=1e-11, eps_f=600., eps_d=300., h_f=50e-9, h_d=1e-9,
    lambda_bottom=5e-9, lambda_top=5e-9)  # None: Neumann; 0: P=0; metres otherwise.
geometry = dict(nx=64, ny=64, nz=32, nd=2, Lx=32e-9, Ly=32e-9)

# 2. Initial field: "negative", "positive", or "near_zero".
# The perturbation settings apply only to near_zero. No mean is constrained later.
initial = dict(kind="near_zero", rms_over_Ps=.01, correlation_length_nm=1.5, seed=7)

# 3. Change the waveform here. All durations are in t0, voltages in V.
# Presets: zero, step_hold, hold_release, triangle, sine, trapezoid, two_step, pund, custom.
waveform = dict(kind="triangle", amplitude_V=.4, hold_t0=40., relaxation_t0=0.,
    period_t0=10., cycles=2, rise_t0=.1, fall_t0=.1, gap_t0=2., two_step_fraction=.45,
    custom_times_t0=[0., 1., 2., 3.], custom_voltages_V=[0., .4, .4, 0.],
    custom_interpolation="linear")  # "hold" gives exact jumps at custom knots.

# 3b. Defects. Pinning sites: local alpha_1 -> alpha1_factor*alpha_1 (only alpha_1 changes)
# at each site center, with a Gaussian profile of radius radius_nm, at count random
# positions inside BTO (fixed seed). They raise the local reversal barrier and impede
# domain-wall motion. count=0 disables them and leaves earlier caches unchanged.
# alpha1_factor and radius_nm are numbers (all sites identical) or [low, high] ranges
# sampled per site; *_dist selects "uniform" or "loguniform". Per-site values are
# saved to <run folder>/sites.json.
# e.g. alpha1_factor=[2., 18.], radius_nm=[.5, 2.], radius_dist="loguniform".
defects = dict(pinning=dict(count=0, alpha1_factor=10., radius_nm=1., seed=11))

# 4. Noise, integration, and saved sampling. noise_over_Ps: small random noise,
# rms noise_over_Ps*Ps per cell per t0 (e.g. 0.01); 0 disables noise.
# Changing any physical/numerical input makes a new cache; plotting changes reuse it.
simulation = dict(noise_over_Ps=.01, seed=42, max_step_t0=.01, output_dt_t0=.05,
                  snapshot_dt_t0=.1, reuse_cache=True)

# 5. Plot/video settings. None selects the middle plane or the whole time range.
# For hold_release, displayed time zero is voltage removal.
view = dict(xy_z_nm=None, xz_y_nm=None, yz_x_nm=None, max_frames=201, fps=12, mp4=True,
            animation_time_range_t0=None, P_limit_over_Ps=1.5)  # e.g. (0, 10) to inspect only early relaxation.

# 6. Optional noise-free continuation at the FINAL voltage, separate from the main run.
# Stationarity within tolerances is not a certificate of the global minimum.
equilibrium = dict(enabled=False, max_t0=600., block_t0=10., max_step_t0=.05,
                   rate_tolerance=1e-3, energy_tolerance=1e-4)


## Shared implementation

The two following cells define the common waveform/runner and plotting functions. Their source is folded for readability. `tdgl3d/solver.py` supplies the verified electrostatic and ETD2/noise solver; it is shared by every preset.

Continuous waveform pieces are integrated without crossing an ideal jump. At a jump the full P field is carried forward, while the electrode charge changes by C_A ΔV. These impulses are stored separately from finite-bin currents. For hold–release plots, displayed time zero is removal; other waveforms start at zero.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class VoltageSegment:
    name: str
    duration_t0: float
    v_start: float
    v_end: float
    kind: str = "linear"

    def callable(self, t0):
        duration = self.duration_t0*t0
        def voltage(t):
            f = np.asarray(t)/duration
            if self.kind == "sine":
                return self.v_start+self.v_end*np.sin(2*np.pi*f)
            return self.v_start+(self.v_end-self.v_start)*f
        return voltage


def build_protocol(w):
    """Continuous pieces, with exact jumps permitted between pieces."""
    kind, a = w["kind"], float(w["amplitude_V"])
    hold, rest = float(w["hold_t0"]), float(w["relaxation_t0"])
    pieces = []
    def add(name, duration, start, end=None, shape="linear"):
        if duration < 0 or not np.isfinite(duration):
            raise ValueError("All durations must be finite and nonnegative.")
        if duration > 0:
            pieces.append(VoltageSegment(name, duration, start,
                                         start if end is None else end, shape))
    event_t0 = None
    if kind == "hold_release" and (hold<=0 or rest<=0):
        raise ValueError("hold_release requires positive hold_t0 and relaxation_t0.")
    if kind == "zero":
        add("zero_voltage", rest, 0)
    elif kind == "step_hold":
        add("step_hold", hold, a)
    elif kind == "hold_release":
        add("voltage_hold", hold, a)
        event_t0 = hold
        add("after_removal", rest, 0)
    elif kind in ["triangle", "sine"]:
        T = float(w["period_t0"])
        for cycle in range(int(w["cycles"])):
            if kind == "sine":
                add(f"cycle_{cycle+1}", T, 0, a, "sine")
            else:
                for j, (v0,v1,d) in enumerate([(0,a,T/4),(a,-a,T/2),(-a,0,T/4)]):
                    add(f"cycle_{cycle+1}_{j+1}",d,v0,v1)
        add("after_waveform", rest, 0)
    elif kind in ["trapezoid", "two_step"]:
        if kind == "two_step":
            b = a*float(w["two_step_fraction"])
            add("first_rise",w["rise_t0"],0,b)
            add("first_hold",hold/2,b)
            add("second_rise",w["rise_t0"],b,a)
            add("second_hold",hold/2,a)
        else:
            add("rise",w["rise_t0"],0,a)
            add("hold",hold,a)
        add("fall",w["fall_t0"],a,0)
        event_t0 = sum(p.duration_t0 for p in pieces)
        add("after_pulse",rest,0)
    elif kind == "pund":
        for label, sign in zip("PUND",[1,1,-1,-1]):
            add(label+"_rise",w["rise_t0"],0,sign*a)
            add(label+"_hold",hold,sign*a)
            add(label+"_fall",w["fall_t0"],sign*a,0)
            add(label+"_gap",w["gap_t0"],0)
    elif kind == "custom":
        times = np.asarray(w["custom_times_t0"],float)
        volts = np.asarray(w["custom_voltages_V"],float)
        if (times.ndim!=1 or len(times)<2 or times.shape!=volts.shape
                or not np.all(np.isfinite(times)) or not np.all(np.isfinite(volts))
                or times[0]!=0 or np.any(np.diff(times)<=0)):
            raise ValueError("Custom knots must start at 0 and increase, with matching finite voltages.")
        interpolation = w["custom_interpolation"]
        if interpolation not in ["linear","hold"]:
            raise ValueError("custom_interpolation must be 'linear' or 'hold'.")
        for i, duration in enumerate(np.diff(times)):
            add(f"custom_{i+1}",float(duration),float(volts[i]),
                float(volts[i+1]) if interpolation=="linear" else float(volts[i]))
        if interpolation=="hold" and rest<=0 and volts[-1]!=volts[-2]:
            raise ValueError("Use relaxation_t0 > 0 to retain the last held-voltage jump.")
        # In hold mode, the last voltage is the terminal value after the last jump.
        # A positive final rest makes that state explicit, even for an ideal final jump.
        add("custom_final_hold",rest,float(volts[-1]))
    else:
        raise ValueError("Unknown waveform: "+kind)
    if not pieces or any(not np.isfinite(p.v_start+p.v_end) for p in pieces):
        raise ValueError("The waveform must have positive duration and finite voltages.")
    if int(w["cycles"]) != w["cycles"] or w["cycles"] < 1:
        raise ValueError("cycles must be a positive integer.")
    return pieces, event_t0


def make_initial_state(model, initial):
    kind = initial["kind"]
    if kind in ["negative", "positive"]:
        return model.uniform_initial(-1 if kind=="negative" else 1)
    if kind != "near_zero":
        raise ValueError("Initial state must be negative, positive, or near_zero.")
    from scipy.ndimage import gaussian_filter
    m,g = model.m,model.g
    rng = np.random.default_rng(initial["seed"])
    field = gaussian_filter(rng.normal(size=(g.nx,g.ny)),
        sigma=(initial["correlation_length_nm"]*1e-9/(g.Lx/g.nx),
               initial["correlation_length_nm"]*1e-9/(g.Ly/g.ny)),mode="wrap")
    field -= field.mean()
    if field.std()==0:
        raise ValueError("Initial perturbation vanished; reduce its correlation length or refine the grid.")
    field *= initial["rms_over_Ps"]*m.Ps/field.std()
    return np.repeat(field[:,:,None],g.nz,axis=2)


def _draw(rng, spec, dist, count, name):
    """A number gives every site that value; [low, high] samples each site."""
    if np.ndim(spec)==0:
        value=float(spec)
        if not np.isfinite(value) or value<=0:raise ValueError(f"{name} must be positive.")
        return np.full(count,value)
    low,high=map(float,spec)
    if not np.isfinite(low+high) or not 0<low<=high:raise ValueError(f"{name} range must satisfy 0 < low <= high.")
    if dist=="uniform":return rng.uniform(low,high,count)
    if dist=="loguniform":return np.exp(rng.uniform(np.log(low),np.log(high),count))
    raise ValueError(f"{name}_dist must be 'uniform' or 'loguniform'.")


def build_defect_sites(material, grid, defects):
    """Random pinning sites: local alpha_1 -> factor*alpha_1 at each center.

    alpha1_factor and radius_nm are numbers (identical sites) or [low, high]
    ranges sampled independently per site with *_dist "uniform" or
    "loguniform". Centers are drawn first, so fixed-size specs are unchanged.
    """
    if not defects:return ()
    pin=defects.get("pinning",{})
    count=int(pin.get("count",0))
    if count<0 or count!=pin.get("count",0):raise ValueError("Pinning-site count must be a nonnegative integer.")
    if count==0:return ()
    rng=np.random.default_rng(pin["seed"])
    centers=rng.uniform([0,0,0],[grid.Lx,grid.Ly,material.h_f],size=(count,3))
    radii=_draw(rng,pin["radius_nm"],pin.get("radius_dist","uniform"),count,"radius_nm")*1e-9
    factors=_draw(rng,pin["alpha1_factor"],pin.get("alpha1_factor_dist","uniform"),count,"alpha1_factor")
    return tuple(NucleationSite(center=tuple(c),radius=(r,)*3,delta_a=(f-1)*material.alpha_1)
                 for c,r,f in zip(centers,radii,factors))


def save_sites(model, out):
    """Per-site centers, radii and alpha_1 factors of a run, for inspection."""
    if not model.sites:return
    rows=[dict(center_nm=[c*1e9 for c in s.center],radius_nm=s.radius[0]*1e9,
               alpha1_factor=1+s.delta_a/model.m.alpha_1) for s in model.sites]
    out.mkdir(parents=True,exist_ok=True)
    (out/"sites.json").write_text(json.dumps(rows,indent=1)+"\n")


def prepare_run(material_settings, geometry, initial, waveform, simulation, cache_root, defects=None):
    material = Material(**material_settings)
    grid = Grid(**geometry)
    sites = build_defect_sites(material,grid,defects)
    model = StackModel(material,grid,sites=sites)
    pieces,event_t0 = build_protocol(waveform)
    if (not np.all(np.isfinite([simulation[k] for k in
            ["noise_over_Ps","max_step_t0","output_dt_t0","snapshot_dt_t0"]]))
            or simulation["noise_over_Ps"]<0
            or min(simulation[k] for k in ["max_step_t0","output_dt_t0","snapshot_dt_t0"])<=0):
        raise ValueError("Use nonnegative finite noise_over_Ps and positive time/snapshot steps.")
    # Cache keys include the solver source, every physical/numerical input, and
    # the built waveform. Plot settings do not force another dynamics run.
    config=dict(material=material_settings,geometry=geometry,initial=initial,
        segments=[asdict(p) for p in pieces],simulation={k:v for k,v in simulation.items()
        if k!="reuse_cache"},solver_sha256=hashlib.sha256((ROOT/"tdgl3d"/"solver.py").read_bytes()).hexdigest(),
        driver_version=1)
    if sites:
        # Only runs with defects carry this key, so earlier defect-free caches keep their hashes.
        config["defects"]=defects
    signature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:12]
    out=Path(cache_root)/f"{waveform['kind']}_{signature}"
    return model,pieces,event_t0,config,out


def combine_phases(phases, pieces, model):
    """Keep both sides of jumps; never form a current bin of zero width."""
    merged={}
    endpoint_keys=["t","V","P","P_std","positive_fraction","E_f","D","dPdt_drift","G_area"]
    current_keys=["t_current","V_current","dPdt","J_total","J_background","J_polarization"]
    clocks=[];current_clocks=[];snapshots=[];snap_t=[];snap_v=[];snap_phase=[]
    phase_ids=[];current_phase=[];jumps=[];offset=0.
    for i,(r,piece) in enumerate(zip(phases,pieces)):
        if i:
            np.testing.assert_allclose(r["P"][0],phases[i-1]["P"][-1],atol=1e-12)
            np.testing.assert_allclose(r["P_snapshots"][0],phases[i-1]["P_final"],rtol=0,atol=5e-8)
            deltaV=float(r["V"][0]-phases[i-1]["V"][-1])
            if abs(deltaV)>1e-12:
                jump=float(r["D"][0]-phases[i-1]["D"][-1])
                np.testing.assert_allclose(jump,model.m.C_area*deltaV,atol=1e-12)
                jumps.append([offset,deltaV,jump])
        clocks.append(r["t"]+offset);current_clocks.append(r["t_current"]+offset)
        phase_ids.extend([i]*len(r["t"]));current_phase.extend([i]*len(r["t_current"]))
        ids=r["snapshot_indices"]
        snapshots.append(r["P_snapshots"]);snap_t.extend(r["t"][ids]+offset)
        snap_v.extend(r["V"][ids]);snap_phase.extend([i]*len(ids))
        offset += float(r["t"][-1])
    for key in endpoint_keys+current_keys:
        if key not in ["t","t_current"]:merged[key]=np.concatenate([r[key] for r in phases])
    merged.update(t=np.concatenate(clocks),t_current=np.concatenate(current_clocks),
        P_snapshots=np.concatenate(snapshots),snapshot_t=np.asarray(snap_t),
        snapshot_V=np.asarray(snap_v),snapshot_phase=np.asarray(snap_phase),
        phase=np.asarray(phase_ids),current_phase=np.asarray(current_phase),
        jumps=np.asarray(jumps,float).reshape(-1,3),P_final=phases[-1]["P_final"])
    return merged


def run_protocol(model,pieces,config,out,initial,simulation):
    manifest=out/"run.json"
    cached=None
    if simulation["reuse_cache"] and manifest.exists():
        cached=json.loads(manifest.read_text())
        if cached.get("config") != config:
            raise RuntimeError("Cache metadata mismatch; use a new run folder.")
    out.mkdir(parents=True,exist_ok=True)
    phases=[]
    P=make_initial_state(model,initial)
    for i,piece in enumerate(pieces):
        file=out/f"phase_{i:02d}.npz"
        if cached is not None and file.exists():
            with np.load(file) as data:r={k:data[k] for k in data.files}
            print(f"Loaded {piece.name}: {piece.duration_t0:g} t0")
        else:
            duration=piece.duration_t0*model.m.t0
            n=max(1,int(np.ceil(piece.duration_t0/simulation["output_dt_t0"]-1e-10)))
            times=np.linspace(0,duration,n+1)
            stride=max(1,int(round(simulation["snapshot_dt_t0"]/(piece.duration_t0/n))))
            print(f"Simulating {piece.name}: {piece.duration_t0:g} t0",flush=True)
            def progress(done,total):
                print(f"  {piece.name}: {100*done/total:.0f}%",flush=True)
            r=model.simulate(piece.callable(model.m.t0),times,P0=P,
                max_step_over_t0=simulation["max_step_t0"],noise_over_Ps=simulation["noise_over_Ps"],
                random_seed=simulation["seed"]+i,snapshot_stride=stride,progress=progress)
            np.savez_compressed(file,**r)
        phases.append(r);P=r["P_final"].copy()
    merged=combine_phases(phases,pieces,model)
    manifest.write_text(json.dumps(dict(config=config,phase_files=[f"phase_{i:02d}.npz" for i in range(len(pieces))],
        phase_names=[p.name for p in pieces]),indent=2)+"\n")
    return phases,merged


def check_result(model,phases,result):
    for r in phases:
        assert all(np.isfinite(v).all() for v in r.values())
        np.testing.assert_allclose(r["J_total"],r["J_background"]+r["J_polarization"],atol=1e-8)
    smooth_charge=sum(float(np.sum(r["J_total"]*np.diff(r["t"]))) for r in phases)
    jump_charge=float(result["jumps"][:,2].sum())
    np.testing.assert_allclose(smooth_charge+jump_charge,
        result["D"][-1]-result["D"][0],atol=1e-11)
    print("Finite fields, continuous P, and total charge including ideal jumps: passed.")


def stationary_check(model,P,voltage,options,out):
    """Optional noise-free local-stationarity diagnostic at the final voltage."""
    if not options["enabled"]:return None
    if min(options[k] for k in ["max_t0","block_t0","max_step_t0","rate_tolerance","energy_tolerance"])<=0:
        raise ValueError("Stationarity-check durations, step, and tolerances must be positive.")
    path=out/"stationary_check.npz";meta=out/"stationary_check.json"
    if meta.exists() and path.exists() and json.loads(meta.read_text())==options:
        with np.load(path) as data:return {k:data[k] for k in data.files}
    m=model.m;last=model.energy_per_area(P,voltage);scale=m.h_f*abs(m.alpha_1)*m.Ps**2
    fields=[P.astype(np.float32)];times=[0.];energy=[last];residual=[];converged=False
    for block in range(int(np.ceil(options["max_t0"]/options["block_t0"]))):
        length=min(options["block_t0"],options["max_t0"]-times[-1]/m.t0)
        if length<=1e-12:break
        r=model.simulate(lambda t:voltage,np.linspace(0,length*m.t0,21),P0=P,
            temperature=0,max_step_over_t0=options["max_step_t0"],snapshot_stride=20)
        P=r["P_final"];E=float(r["G_area"][-1]);rate=m.t0/m.Ps*np.sqrt(np.mean(model.drift(P,voltage)**2))
        if np.max(np.diff(r["G_area"]))>1e-8:
            raise RuntimeError("Stationarity-check energy increased; reduce its max_step_t0.")
        change=abs(E-last)/scale;last=E
        fields.append(P.astype(np.float32));times.append(times[-1]+length*m.t0);energy.append(E);residual.append(rate)
        print(f"Noise-free check t/t0={times[-1]/m.t0:g}: residual={rate:.3e}",flush=True)
        if rate<options["rate_tolerance"] and change<options["energy_tolerance"]:
            converged=True;break
    answer=dict(P_final=P,P_snapshots=np.asarray(fields),t=np.asarray(times),G_area=np.asarray(energy),
                residual=np.asarray(residual),converged=np.array(converged))
    np.savez_compressed(path,**answer);meta.write_text(json.dumps(options,indent=2)+"\n")
    return answer


In [ ]:
def plot_diagnostics(result,model,event_t0,out,waveform_kind,protocol=None,phase_results=None):
    m=model.m;origin=0. if event_t0 is None else event_t0*m.t0
    tau=(result["t"]-origin)/m.t0
    fig,axes=plt.subplots(5,1,figsize=(10,11),sharex=True,layout="constrained")
    for ax,key,label in zip(axes,["V","P","P_std","positive_fraction","G_area"],
            ["V (V)","Mean P (C/m²)","Spatial std P (C/m²)","Fraction P > 0","G / area (J/m²)"]):
        ax.plot(tau,result[key]);ax.set_ylabel(label)
        if event_t0 is not None:ax.axvline(0,color="black",ls="--",lw=.8)
    axes[-1].set_xlabel("Time since voltage removal / t0" if event_t0 is not None else "Time / t0")
    fig.savefig(out/"diagnostics.png",dpi=130);fig.savefig(out/"diagnostics.pdf");plt.show()
    fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True,layout="constrained")
    clock=(result["t_current"]-origin)/m.t0
    axes[0].plot(clock,result["dPdt"]);axes[0].set_ylabel("Finite-bin d(mean P)/dt (A/m²)")
    axes[1].plot(clock,result["J_total"],label="Terminal current")
    axes[1].plot(clock,result["J_polarization"],label="Polarization contribution",alpha=.7)
    axes[1].set(xlabel="Time since voltage removal / t0" if event_t0 is not None else "Time / t0",ylabel="Current density (A/m²)")
    axes[1].legend();fig.savefig(out/"rates.png",dpi=130);fig.savefig(out/"rates.pdf");plt.show()
    if event_t0 is not None:
        mask=result["t"]>origin;bins=result["t_current"]>origin
        fig,axes=plt.subplots(2,1,figsize=(9,6),sharex=True,layout="constrained")
        axes[0].semilogx(tau[mask],result["P"][mask]);axes[0].set_ylabel("Mean P (C/m²)")
        axes[1].semilogx(clock[bins],result["dPdt"][bins]);axes[1].set(
            xlabel="Time since voltage removal / t0",ylabel="Finite-bin d(mean P)/dt (A/m²)")
        fig.savefig(out/"post_release.png",dpi=130);fig.savefig(out/"post_release.pdf");plt.show()
    if len(result["jumps"]):
        print("Ideal voltage jumps (their charge impulses are separate from finite-bin currents):")
        for t,dV,dD in result["jumps"]:
            print(f"  t/t0={(t-origin)/m.t0:g}: ΔV={dV:g} V, ΔD={dD:.6g} C/m²")
    if waveform_kind=="pund":
        if protocol is None or phase_results is None:raise ValueError("PUND needs its protocol and phase data.")
        # Each identically sampled block contains rise, hold, fall, and gap.
        sub={};blocks=[]
        for label in "PUND":
            ids=[i for i,p in enumerate(protocol) if p.name.startswith(label+"_")]
            pick=np.isin(result["current_phase"],ids)
            blocks.append((result["J_total"][pick],result["t_current"][pick]))
        if len({len(J) for J,t in blocks})!=1:raise RuntimeError("PUND blocks have unequal sampling.")
        Jpu=blocks[0][0]-blocks[1][0];Jnd=blocks[2][0]-blocks[3][0]
        ids=[i for i,p in enumerate(protocol) if p.name.startswith("P_")]
        widths=np.concatenate([np.diff(phase_results[i]["t"]) for i in ids])
        t=blocks[0][1]
        fig,axes=plt.subplots(2,1,figsize=(9,6),sharex=True,layout="constrained")
        for label,J in [("P − U",Jpu),("N − D",Jnd)]:
            axes[0].plot(t/m.t0,J,label=label);axes[1].plot(t/m.t0,np.cumsum(J*widths),label=label)
        axes[0].set_ylabel("Subtracted J (A/m²)");axes[0].legend()
        axes[1].set(xlabel="Time within pulse block / t0",ylabel="Subtracted charge / area (C/m²)")
        fig.savefig(out/"pund.png",dpi=130);fig.savefig(out/"pund.pdf");plt.show()
        np.savez_compressed(out/"pund_subtracted.npz",t=t,J_PU=Jpu,J_ND=Jnd,
            delta_D_PU=np.cumsum(Jpu*widths),delta_D_ND=np.cumsum(Jnd*widths))


def select_animation_frames(result,model,event_t0,view):
    origin=0. if event_t0 is None else event_t0*model.m.t0
    tau=(result["snapshot_t"]-origin)/model.m.t0
    eligible=np.arange(len(tau))
    if view["animation_time_range_t0"] is not None:
        low,high=view["animation_time_range_t0"]
        if not np.isfinite(low+high) or low>=high:raise ValueError("Use an increasing finite animation time range.")
        eligible=eligible[(tau>=low-1e-10)&(tau<=high+1e-10)]
    if len(eligible)<2:raise ValueError("Animation range must include at least two saved frames.")
    if view["max_frames"]<2 or view["fps"]<=0:raise ValueError("Use max_frames >= 2 and fps > 0.")
    boundaries=np.flatnonzero(np.diff(result["snapshot_phase"])!=0)
    required=np.unique(np.r_[eligible[0],eligible[-1],boundaries,boundaries+1])
    if event_t0 is not None:
        required=np.unique(np.r_[required,np.flatnonzero((tau>=-1e-10)&(tau<=5+1e-10))])
    required=np.intersect1d(required,eligible)
    budget=max(0,int(view["max_frames"])-len(required))
    remaining=np.setdiff1d(eligible,required)
    selected=remaining[np.unique(np.linspace(0,len(remaining)-1,min(budget,len(remaining)),dtype=int))] if budget and len(remaining) else []
    frames=np.unique(np.r_[required,selected]).astype(int)
    return frames,origin


def build_animation_fields(result,model,event_t0,view,out):
    m,g=model.m,model.g
    z=m.h_f*.5 if view["xy_z_nm"] is None else view["xy_z_nm"]*1e-9
    y=g.Ly*.5 if view["xz_y_nm"] is None else view["xz_y_nm"]*1e-9
    if not (np.isfinite(z+y) and 0<=z<=m.h_f and 0<=y<=g.Ly):
        raise ValueError("Choose an xy cut inside BTO and an xz cut inside the lateral box.")
    zi=g.nz//2 if view["xy_z_nm"] is None else int(np.argmin(abs(model.z-z)))
    yi=g.ny//2 if view["xz_y_nm"] is None else int(np.argmin(abs(model.y-y)))
    ids,origin=select_animation_frames(result,model,event_t0,view)
    fields={k:[] for k in ["P_xy","P_xz","Ez_xy","Ez_xz","phi_xy","phi_xz"]}
    for i in ids:
        P=result["P_snapshots"][i].astype(float);e=model.electrostatics(P,float(result["snapshot_V"][i]))
        Ps=np.zeros_like(e["phi"]);Ps[...,model.fe]=P
        E=(.5*(e["D_faces"][...,:-1]+e["D_faces"][...,1:])-Ps)/model.eps
        np.testing.assert_allclose(E[...,model.fe],e["E_z"],rtol=1e-10,atol=1e-6)
        for key,array,z_index in [("P",P,zi),("Ez",E,model.fe.start+zi),("phi",e["phi"],model.fe.start+zi)]:
            fields[key+"_xy"].append(array[:,:,z_index].T)
            fields[key+"_xz"].append(array[:,yi,:].T)
    fields={k:np.asarray(v) for k,v in fields.items()}
    fields.update(frame_ids=ids,t=result["snapshot_t"][ids],V=result["snapshot_V"][ids],
        origin_s=np.array(origin),z_faces_m=model.z_faces,xy_z_m=np.array(model.z[zi]),xz_y_m=np.array(model.y[yi]))
    np.savez_compressed(out/"animation_fields.npz",**fields)
    return fields


def render_animation(fields,model,view,out):
    m,g=model.m,model.g
    fig,axes=plt.subplots(3,2,figsize=(12,14.5))
    fig.subplots_adjust(left=.075,right=.78,bottom=.045,top=.935,wspace=.42,hspace=.38)
    xf=np.linspace(0,g.Lx*1e9,g.nx+1);zf=model.z_faces*1e9
    h,d,L=m.h_f*1e9,m.h_d*1e9,g.Lx*1e9
    extent=[0,L,0,g.Ly*1e9]
    Pmax=view["P_limit_over_Ps"]*m.Ps
    if not np.isfinite(Pmax) or Pmax<=0:raise ValueError("P_limit_over_Ps must be positive and finite.")
    limits=[Pmax,max(np.max(abs(fields["Ez_xy"])),np.max(abs(fields["Ez_xz"])),1.)*1e-6,
            max(np.max(abs(fields["phi_xy"])),np.max(abs(fields["phi_xz"])),1e-12)*1e3]
    labels=["P in BTO (C/m²); gray = dielectric","E_z (MV/m)","φ (mV; top electrode at 0)"]
    artists=[];bottom=[]
    for row,(quantity,factor) in enumerate([("P",1.),("Ez",1e-6),("phi",1e3)]):
        limit=limits[row]
        im=axes[row,0].imshow(fields[quantity+"_xy"][0]*factor,origin="lower",cmap="RdBu_r",
            vmin=-limit,vmax=limit,extent=extent,aspect="equal")
        mesh=axes[row,1].pcolormesh(xf,zf[model.fe.start:model.fe.stop+1] if row==0 else zf,
            fields[quantity+"_xz"][0]*factor,cmap="RdBu_r",vmin=-limit,vmax=limit,shading="flat")
        artists.append((im,mesh))
        name=["P","E_z","φ"][row]
        axes[row,0].set(xlabel="x (nm)",ylabel="y (nm)",
            title=f"{name}: x–y inside BTO, z={float(fields['xy_z_m'])*1e9:.2f} nm")
        ax=axes[row,1];ax.set_aspect("equal")
        ax.set(xlabel="x (nm)",ylabel="z (nm; BTO starts at 0)",xlim=(0,L),ylim=(-d,h+d),
            title=f"{name}: full stack, y={float(fields['xz_y_m'])*1e9:.2f} nm")
        if d>0:
            for boundary in [0,h]:ax.axhline(boundary,color="black",ls="--",lw=.8)
            if row==0:
                for low,high in [(-d,0),(h,h+d)]:
                    ax.axhspan(low,high,facecolor=".7",hatch="////",edgecolor=".4",lw=0)
            ax.annotate(f"D2: {d:g} nm\nelectrode: 0 V",xy=(L,h+d/2),xytext=(8,0),
                textcoords="offset points",ha="left",va="top",fontsize=8,annotation_clip=False)
            label=f"D1: {d:g} nm\nelectrode: {fields['V'][0]:g} V"
        else:
            ax.annotate("electrode: 0 V",xy=(L,h),xytext=(8,0),textcoords="offset points",
                ha="left",va="top",fontsize=8,annotation_clip=False)
            label=f"electrode: {fields['V'][0]:g} V"
        bottom.append(ax.annotate(label,xy=(L,-d/2),xytext=(8,0),textcoords="offset points",
            ha="left",va="bottom",fontsize=8,annotation_clip=False))
        ax.text(1.04,.5,f"BTO\n{h:g} nm",transform=ax.transAxes,ha="left",va="center",fontsize=8)
        for boundary in [-d,h+d]:ax.axhline(boundary,color="black",lw=2,clip_on=False)
        box=ax.get_position();cax=fig.add_axes([.91,box.y0+.07*box.height,.015,.86*box.height])
        fig.colorbar(im,cax=cax,label=labels[row])
    for ax in axes.ravel():ax.grid(False)
    title=fig.suptitle("",y=.98)
    def update(position):
        for row,(key,factor) in enumerate([("P",1.),("Ez",1e-6),("phi",1e3)]):
            artists[row][0].set_data(fields[key+"_xy"][position]*factor)
            artists[row][1].set_array((fields[key+"_xz"][position]*factor).ravel())
            bottom[row].set_text((f"D1: {d:g} nm\n" if d>0 else "")+f"electrode: {fields['V'][position]:g} V")
        tau=(fields["t"][position]-float(fields["origin_s"]))/m.t0
        title.set_text(f"V={fields['V'][position]:g} V; displayed t/t0={tau:.2f} | dielectric / BTO / dielectric" if d>0
                       else f"V={fields['V'][position]:g} V; displayed t/t0={tau:.2f} | direct electrodes / BTO")
        return [artist for pair in artists for artist in pair]+bottom+[title]
    update(0);fig.canvas.draw();fig.set_layout_engine("none")
    movie=FuncAnimation(fig,update,frames=len(fields["t"]),interval=1000/view["fps"],blit=False)
    # Guard against silent frame truncation for larger user-selected videos.
    with plt.rc_context({"animation.embed_limit":max(40.,len(fields["t"])*.8)}):
        html=movie.to_jshtml(fps=view["fps"])
    update(len(fields["t"])-1);fig.savefig(out/"final_fields.png",dpi=120)
    plt.close(fig)
    (out/"animation.html").write_text("<!DOCTYPE html><meta charset='utf-8'><title>3D TDGL waveform simulation</title>"+html)
    display(HTML(html))
    return html


def plot_stationarity(check,model,out):
    if check is None:return
    m=model.m
    fig,axes=plt.subplots(1,2,figsize=(10,4),layout="constrained")
    axes[0].plot(check["t"]/m.t0,check["G_area"]);axes[0].set(xlabel="Noise-free time / t0",ylabel="G / area (J/m²)")
    axes[1].semilogy(check["t"][1:]/m.t0,check["residual"],"o-")
    axes[1].set(xlabel="Noise-free time / t0",ylabel="Normalized deterministic RMS rate")
    fig.savefig(out/"stationarity.png",dpi=130);plt.show()
    print("Local stationarity criteria met:",bool(check["converged"]))


def plot_histograms(result,model,out):
    snapshots=result["P_snapshots"]
    frames=np.unique(np.linspace(0,len(snapshots)-1,5,dtype=int))
    low=float(snapshots.min());high=float(snapshots.max())
    edges=np.linspace(low-1e-9,high+1e-9,101)
    fig,ax=plt.subplots(figsize=(8,4),layout="constrained")
    for frame in frames:
        ax.hist(snapshots[frame].ravel(),bins=edges,density=True,histtype="step",
            label=f"t/t0={result['snapshot_t'][frame]/model.m.t0:.1f}")
    ax.set(xlabel="Local P (C/m²)",ylabel="Probability density");ax.legend(fontsize=8)
    fig.savefig(out/"histograms.png",dpi=130);fig.savefig(out/"histograms.pdf");plt.show()


## Run the chosen waveform

Each run folder has its exact configuration and independent phase files. Matching caches are reused only when geometry, material, initialization, waveform, seeds, solver source, and integration/sampling settings agree. Set `reuse_cache=False` to recompute.


In [ ]:
model, protocol, event_t0, config, OUT = prepare_run(
    material_settings, geometry, initial, waveform, simulation, ROOT/"runs", defects)
print(f"Ps={model.m.Ps:.6f} C/m²; t0={model.m.t0*1e6:.3f} µs")
print("Run folder:", Path("runs")/OUT.name)
save_sites(model, OUT)
phase_results, result = run_protocol(model, protocol, config, OUT, initial, simulation)
check_result(model, phase_results, result)


## Voltage, polarization, rate, and energy

A near-zero spatial mean can coexist with ordered opposite-sign domains. The histogram and spatial standard deviation help distinguish them. PUND additionally plots matched P−U and N−D currents and integrated charge. The current curves show finite-bin averages; ideal voltage-jump impulses are printed separately. Under noise the instantaneous energy need not decrease monotonically.


In [ ]:
plot_diagnostics(result, model, event_t0, OUT, waveform["kind"], protocol, phase_results)
plot_histograms(result, model, OUT)


## Synchronized orthogonal field animations

Columns show x–y, x–z, and y–z sections. The initial rows are P_z, E_z, and φ; choose E_x, E_y, or |E| from the field menu, or enable all components. Playback, frame stepping, speed, and the time slider are synchronized across every panel. The viewer also saves the current view as a PNG. Select **BTO contrast** to show weaker fields inside BTO; dielectric values outside that E range are visibly clipped at the labeled limits.

`xy_z_nm`, `xz_y_nm`, and `yz_x_nm` select each plane; `None` chooses the central cell. Vertical sections include both dielectric layers at their actual thicknesses. Gray in P panels means no ferroelectric order parameter in the dielectric. Electrical fields span the full stack, with BTO interfaces and electrode voltages labeled. Each quantity has a fixed color scale across all planes and selected frames.

E_x and E_y use periodic Fourier derivatives of φ. E_z uses the conservative face-displacement reconstruction and is checked against the solver inside BTO. φ is referenced to the grounded top electrode. The standalone HTML uses 16-bit display quantization; With `view["mp4"]=True`, `cross_sections/cross_sections.mp4` also shows P_z, E_z, |E|, and φ in all three planes (E and φ clipped at the 99.5th percentile, marked by colorbar arrows). `cross_sections/orthogonal_fields.npz` retains unquantized float32 sections in SI units. Set `animation_time_range_t0=(0,10)` to inspect early dynamics using scales for that time window.


In [ ]:
from tdgl3d.animations import build_orthogonal_fields, write_field_viewers, write_field_movie
frame_ids, animation_origin = select_animation_frames(result, model, event_t0, view)
animation_fields = build_orthogonal_fields(result, model, view, frame_ids, animation_origin)
animation_path = write_field_viewers(animation_fields, model, OUT/"cross_sections", fps=view["fps"])
animation_html = animation_path.read_text()
display(HTML(animation_html))
print("Saved:", animation_path.relative_to(ROOT))

if view.get("mp4", False):
    movie_path = write_field_movie(animation_fields, model, OUT/"cross_sections"/"cross_sections.mp4",
        fps=view["fps"], title=f"{waveform['kind']} · noise_over_Ps = {simulation['noise_over_Ps']:g}", traces=result)
    print("Saved:", movie_path.relative_to(ROOT))


## Optional check of local stationarity

Enable `equilibrium["enabled"]` to continue the final state without noise at the final voltage. This diagnostic leaves the main waveform trajectory unchanged. It stops when both the normalized deterministic RMS rate and block energy change meet their tolerances, or reports that the time limit was reached. It does not establish the global minimum.


In [ ]:
settled = stationary_check(model, result["P_final"].copy(),
    float(result["V"][-1]), equilibrium, OUT)
plot_stationarity(settled, model, OUT)


## Saved summary

Phase NPZ files contain SI-unit traces and 3D P snapshots. `animation_fields.npz` contains the selected P/E_z/φ slices and their physical coordinates. `animation.html` opens independently of Jupyter. These are illustrative device parameters; quantitative switching times and tail amplitudes require experimental calibration and time-step/coarse-graining checks.


In [ ]:
summary = dict(waveform=waveform["kind"], configuration=config,
    initial_mean_P_C_per_m2=float(result["P"][0]), final_mean_P_C_per_m2=float(result["P"][-1]),
    final_spatial_std_C_per_m2=float(result["P_std"][-1]),
    final_positive_fraction=float(result["positive_fraction"][-1]),
    displayed_time_origin_t0=event_t0, voltage_jumps_t_deltaV_deltaD=result["jumps"].tolist(),
    animation_frames=len(animation_fields["t"]),
    stationary_check_enabled=equilibrium["enabled"],
    stationary_check_converged=None if settled is None else bool(settled["converged"]))
(OUT/"summary.json").write_text(json.dumps(summary, indent=2)+"\n")
print(f"Mean P: {result['P'][0]:.6f} → {result['P'][-1]:.6f} C/m²")
print(f"Final fraction P>0: {result['positive_fraction'][-1]:.6f}")
print("All data and figures:", Path("runs")/OUT.name)
